In [8]:
import numpy as np
from typing import Callable, List, Tuple
import random


# ====================== N-QUEENS FITNESS FUNCTION ======================
def n_queens_fitness(chromosome: np.ndarray) -> int:
    """Count diagonal conflicts in an N-Queens solution."""
    conflicts = 0
    n = len(chromosome)
    for i in range(n):
        for j in range(i + 1, n):
            if abs(chromosome[i] - chromosome[j]) == abs(i - j):
                conflicts += 1
    return conflicts


# ====================== CROSSOVER (ORDER-1 FOR PERMUTATIONS) ======================
def order1_crossover(parent1: np.ndarray, parent2: np.ndarray) -> np.ndarray:
    """Order-1 crossover for permutations."""
    n = len(parent1)
    start, end = sorted(random.sample(range(n), 2))
    child = np.zeros(n, dtype=int)
    child[start:end] = parent1[start:end]
    remaining = [gene for gene in parent2 if gene not in child]
    child[:start] = remaining[:start]
    child[end:] = remaining[start:]
    return child


# ====================== MUTATION (SWAP TWO GENES) ======================
def swap_mutation(chromosome: np.ndarray, mutation_rate: float = 0.1) -> np.ndarray:
    """Swap two random positions in the chromosome."""
    if random.random() < mutation_rate:
        i, j = random.sample(range(len(chromosome)), 2)
        chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
    return chromosome


# ====================== INITIALIZE PERMUTATION POPULATION ======================
def initialize_population(population_size: int, n: int) -> np.ndarray:
    """Generate random permutations for N-Queens."""
    return np.array([np.random.permutation(n) + 1 for _ in range(population_size)])


# ====================== MODIFIED GENETIC ALGORITHM ======================
def n_queens_ga(
        n: int,
        population_size: int = 100,
        max_generations: int = 1000,
        mutation_rate: float = 0.1,
        elitism: bool = True
) -> Tuple[np.ndarray, int]:
    """Solve N-Queens using a genetic algorithm."""
    population = initialize_population(population_size, n)
    best_solution = None
    best_fitness = float('inf')

    for generation in range(max_generations):
        # Evaluate fitness
        fitness = np.array([n_queens_fitness(ind) for ind in population])
        best_idx = np.argmin(fitness)
        current_best_fitness = fitness[best_idx]

        if current_best_fitness < best_fitness:
            best_solution = population[best_idx]
            best_fitness = current_best_fitness
            if best_fitness == 0:  # Early exit if solution found
                break

        # Elitism: Keep top 10% of the population
        next_generation = []
        if elitism:
            elite_size = max(1, population_size // 10)
            elite_indices = np.argsort(fitness)[:elite_size]
            next_generation.extend(population[elite_indices])

        # Fill the rest with crossover + mutation
        while len(next_generation) < population_size:
            # Tournament selection
            candidates = np.random.choice(population_size, size=3, replace=False)
            parent1, parent2 = population[candidates[0]], population[candidates[1]]

            # Crossover and mutation
            child = order1_crossover(parent1, parent2)
            child = swap_mutation(child, mutation_rate)
            next_generation.append(child)

        population = np.array(next_generation)

    return best_solution, best_fitness

In [9]:
def print_solution(solution: np.ndarray) -> None:
    """Print the N-Queens solution as a tuple and a nicely formatted chessboard."""
    n = len(solution)
    print(f"\nSolution tuple: ", solution)
    print("Chessboard:\n")

    # Column headers (A, B, C, ...)
    col_labels = '   ' + ' '.join(chr(ord('A') + i) for i in range(n))
    print(col_labels)
    print('  +' + '--' * n + '+')

    # Board with queens (♛) and dots
    for row in range(n):
        line = f"{row+1:2}|"
        for col in range(n):
            queen_row = solution[col] - 1  # 1-based to 0-based
            line += '♛ ' if queen_row == row else '. '
        line += '|'
        print(line)

    print('  +' + '--' * n + '+\n')




# Solve for N=8
solution, conflicts = n_queens_ga(n=8)
print_solution(solution)


Solution tuple:  [6 3 7 4 1 8 2 5]
Chessboard:

   A B C D E F G H
  +----------------+
 1|. . . . ♛ . . . |
 2|. . . . . . ♛ . |
 3|. ♛ . . . . . . |
 4|. . . ♛ . . . . |
 5|. . . . . . . ♛ |
 6|♛ . . . . . . . |
 7|. . ♛ . . . . . |
 8|. . . . . ♛ . . |
  +----------------+



In [10]:
def get_conflict_map(chromosome: np.ndarray) -> np.ndarray:
    """Returns an array of conflict counts for each queen (column)."""
    n = len(chromosome)
    conflict_map = np.zeros(n, dtype=int)
    for i in range(n):
        for j in range(i + 1, n):
            if abs(chromosome[i] - chromosome[j]) == abs(i - j):
                conflict_map[i] += 1
                conflict_map[j] += 1
    return conflict_map

def conflict_directed_mutation(chromosome, conflict_map):
    conflicted_queens = [i for i, conflicts in enumerate(conflict_map) if conflicts > 0]
    if conflicted_queens:
        i, j = random.sample(conflicted_queens, 2)
        chromosome[i], chromosome[j] = chromosome[j], chromosome[i]
    return chromosome


def hill_climbing(solution):
    current_fitness = n_queens_fitness(solution)
    for i in range(len(solution)):
        for j in range(i+1, len(solution)):
            solution[i], solution[j] = solution[j], solution[i]
            new_fitness = n_queens_fitness(solution)
            if new_fitness >= current_fitness:
                solution[i], solution[j] = solution[j], solution[i]  # Revert
            else:
                current_fitness = new_fitness
    return solution




def tournament_selection(population: np.ndarray, fitness: np.ndarray, k: int = 3):
    selected_indices = np.random.choice(len(population), k, replace=False)
    best_index = selected_indices[np.argmin(fitness[selected_indices])]
    return population[best_index]

# ====================== ADAPTIVE MUTATION RATE ======================
def adaptive_mutation_rate(population: np.ndarray, base_rate: float) -> float:
    diversity = np.mean([len(set(ind)) for ind in population])
    if diversity < len(population[0]) * 0.5:
        return min(1.0, base_rate * 2)
    return base_rate


In [11]:
def hybrid_ga(
    n: int,
    population_size: int = 100,
    max_generations: int = 500,
    base_mutation_rate: float = 0.1,
    elitism_ratio: float = 0.1
) -> Tuple[np.ndarray, int]:

    population = initialize_population(population_size, n)
    best_solution, best_fitness = None, float('inf')
    no_improvement = 0

    for generation in range(max_generations):
        print(f"Generation {generation}")
        fitness = np.array([n_queens_fitness(ind) for ind in population])
        conflict_maps = [get_conflict_map(ind) for ind in population]

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < best_fitness:
            best_solution = population[current_best_idx].copy()
            best_fitness = fitness[current_best_idx]
            no_improvement = 0
        else:
            no_improvement += 1

        if best_fitness == 0 or no_improvement > 50:
            break

        elite_size = int(elitism_ratio * population_size)
        elite_indices = np.argsort(fitness)[:elite_size]
        next_generation = [population[i].copy() for i in elite_indices]

        mutation_rate = adaptive_mutation_rate(population, base_mutation_rate)

        while len(next_generation) < population_size:
            parent1 = tournament_selection(population, fitness)
            parent2 = tournament_selection(population, fitness)
            child = order1_crossover(parent1, parent2)

            # Conflict-directed mutation
            child_idx = np.where((population == parent1).all(axis=1))[0]
            if len(child_idx) > 0:
                parent1_idx = child_idx[0]
                child = conflict_directed_mutation(child, conflict_maps[parent1_idx])
            else:
                child = swap_mutation(child, mutation_rate)

            next_generation.append(child)

        # Hill climbing on best solution
        if generation % 5 == 0:
            next_generation[0] = hill_climbing(next_generation[0])

        # Restart part of population
        if no_improvement > 10:
            next_generation[-20:] = initialize_population(20, n)

        population = np.array(next_generation)

    return best_solution, best_fitness

In [12]:
solution, conflicts = hybrid_ga(n=40)
print_solution(solution)

Generation 0
Generation 1
Generation 2
Generation 3
Generation 4
Generation 5
Generation 6
Generation 7
Generation 8
Generation 9
Generation 10
Generation 11
Generation 12
Generation 13
Generation 14
Generation 15
Generation 16
Generation 17
Generation 18
Generation 19
Generation 20
Generation 21
Generation 22
Generation 23
Generation 24
Generation 25
Generation 26
Generation 27
Generation 28
Generation 29
Generation 30
Generation 31
Generation 32
Generation 33
Generation 34
Generation 35
Generation 36
Generation 37
Generation 38
Generation 39
Generation 40
Generation 41

Solution tuple:  [35 22 14 40 23 27 39 21  9  2  5  1  8 24 12  7 26 34 18  6 11  3 19 25
 33 30 32 13 31 36 38 15 20 16  4 29 17 10 37 28]
Chessboard:

   A B C D E F G H I J K L M N O P Q R S T U V W X Y Z [ \ ] ^ _ ` a b c d e f g h
  +--------------------------------------------------------------------------------+
 1|. . . . . . . . . . . ♛ . . . . . . . . . . . . . . . . . . . . . . . . . . . . |
 2|. . . . . . 